# BEP

# Pre-work

In [39]:
# imports

from pathlib import Path
import pandas as pd
import json

In [68]:
# constants

ROOT_DIR = Path.cwd().parent
DATASET_DIR = ROOT_DIR / "datasets"

NOVIELLI_GOLD_STANDARD_DIR = DATASET_DIR / "novielli-gold-standard"
GH_GOLD_STANDARD = NOVIELLI_GOLD_STANDARD_DIR / "Github Gold Standard.csv"
SO_GOLD_STANDARD = NOVIELLI_GOLD_STANDARD_DIR / "Stack Overflow Gold Standard.csv"

COUTINHO_DIR = DATASET_DIR / "coutinho-sentiment-replication"
COUTINHO = COUTINHO_DIR / "dataset.json"

DEVA_INPUT_DIR = ROOT_DIR / "input" / "deva"
DEVA_OUTPUT_DIR = ROOT_DIR / "output" / "deva"

### Data collection (Novielli et al., 2021)

The dataset used by Novieli et al. is in CSV format, which makes it straight forward to create dataframes out of it.

In [67]:
df_gs_gh = pd.read_csv(GH_GOLD_STANDARD)
# removes trailing `"`
df_gs_gh['Text'] = df_gs_gh['Text'].str.rstrip('"')

df_gs_so = pd.read_csv(SO_GOLD_STANDARD)

In [105]:
print("GH Gold Standard")
df_gs_gh.head(5)

GH Gold Standard


,Type,ID,Text,Gold label
0,commit comments no security,329914,Hmm... Very mysterious development. Not be eas...,negative
1,commit comments no security,2728032,:+1:,neutral
2,commit comments no security,3098507,So what I'm hearing is that if I: * tag consis...,neutral
3,commit comments no security,3103230,How did you tested by normal player? Have you ...,positive
4,commit comments no security,95688,Nice indeed. An earlier version of this patch ...,positive


In [106]:
print("SO Gold Standard")
df_gs_so.head(5)

SO Gold Standard


,ID,Text,Gold label
0,25607311,I saw a really nice post on stackoverflow on t...,positive
1,25386947,I tried so many different ways for printing wi...,negative
2,25419608,I have a general style definition as Now want ...,neutral
3,25727724,My category.php uses wp_list_categories to dis...,neutral
4,25346200,I'm trying to test my controller which deal wi...,neutral


### Data collection (Coutinho et al., 2024)

Since this dataset is in nested JSON format, we do some extra pre-processing to normalize it.

In [100]:
text = COUTINHO.read_text(encoding="utf-8")

if text.lstrip().startswith("["):
    data = json.loads(text)
else:
    try:
        data = [json.loads(line) for line in text.splitlines() if line.strip()]
    except json.JSONDecodeError:
        data = [json.loads(text)]

df_cou = pd.json_normalize(data, sep="_")

df_cou['gold_label'] = df_cou['part2_aggregate_polarity'].where(
    df_cou['part2_aggregate_polarity'] != 'undefined',
    other=df_cou['discussion_polarity']
)
df_cou = df_cou[df_cou['gold_label'].notna() & (df_cou['gold_label'] != 'undefined')]

In [107]:
print(df_cou.shape)
df_cou.head(5)

(1791, 13)


,raw_message,clean_message,message_url,part2_aggregate_polarity,part2_aggregate_avg_confidence,part2_aggregate_agreement_type,tools_SentiStrength,tools_SentiStrengthSE,tools_SentiCR,tools_DEVA,tools_Senti4SD,discussion_polarity,gold_label
0,@snicoll did you get a chance to follow up on ...,did you get a chance to follow up on the issue...,https://github.com/spring-projects/spring-boot...,neutral,4.333333,neuro_and_comp,neutral,neutral,neutral,neutral,neutral,NaN,neutral
1,Thank you for this PR. This is an interesting...,Thank you for this PR. This is an interesting...,https://github.com/google/guice/pull/1349#issu...,undefined,4.666667,undefined,negative,negative,negative,negative,neutral,neutral,neutral
2,"@mbhave, oh sorry for this, my fault, i'll be...",", oh sorry for this, my fault, i'll be more c...",https://github.com/spring-projects/spring-boot...,negative,5.000000,all,negative,negative,neutral,negative,neutral,NaN,negative
3,I've added `merge-with-amendments` so that we ...,I've added so that we can add documentation r...,https://github.com/spring-projects/spring-boot...,neutral,4.000000,comp_only,neutral,neutral,neutral,neutral,neutral,NaN,neutral
4,> @Primary and @ConditionalOnMissingBean. I th...,Applications can still use or qualifiers to g...,https://github.com/spring-projects/spring-boot...,negative,4.666667,all,negative,negative,neutral,negative,negative,NaN,negative


# DEVA Experiments

## Running DEVA collected datasets

DEVA expects an input .txt file with the message to be analyzed and its ID. To demonstrate, this is the example input they provide with the tool:
```
58204 Yes, that's correct.
32957 Challenge accepted! :)
45982 No problem.  It was my pleasure, and thanks for all the help.
63699 Still need tests  Neutral
14350 My mistake, sorry
```

It outputs the original input, as well as three extra columns: its valence score, its arousal score, and an emotional state label derived from the combination of both.

For the purposes of this replication study, however, the benchmark reduces sentiment to a three-class polarity task (positive, negative, neutral), so only the valence score is used. This maps directly to Novielli et al. replication's $±[1,3]$ schema:
- valence > +1 -> positive
- valence < -1 -> negative
- valence = ± 1 -> neutral

The output for the example input looks as follows:
```
58204 Yes, that's correct.  -2  -3    DEPRESSED
32957 Challenge accepted! :)  2   5   EXCITED
45982 No problem.  It was my pleasure, and thanks for all the help.   4   3   EXCITED
63699 Still need tests  Neutral   1   -2    NEUTRAL (LA)
14350 My mistake, sorry   -2  1   NEG VALENCE
```

Therefore, to run DEVA on the datasets collected above, we need to generate valid input, and then parse the output into dataframes.

In [64]:
# df_gs_gh
df_gs_gh[['ID', 'Text']].drop_duplicates(subset='ID').to_csv(
    DEVA_INPUT_DIR / "gs_gh.txt",
    sep='\t', header=False, index=False
)

In [65]:
#df_gs_so
df_gs_so[['ID', 'Text']].drop_duplicates(subset='ID').to_csv(
    DEVA_INPUT_DIR / "gs_so.txt",
    sep='\t', header=False, index=False
)

In [66]:
df_cou_clean = df_cou[['clean_message']].copy()
df_cou_clean.insert(0, 'ID', range(len(df_cou_clean)))
df_cou_clean.drop_duplicates(subset='ID').to_csv(
    DEVA_INPUT_DIR / "cou.txt",
    sep='\t', header=False, index=False
)

In [72]:
def parse_deva_output(filepath, dataset_label):
    rows = []
    with open(filepath, 'r') as f:
        for line in f:
            parts = line.rstrip('\n').rsplit('\t', 3)  # split from right: [id+text, valence, arousal, label]
            if len(parts) == 4:
                id_and_text, valence, arousal, label = parts
                id_, text = id_and_text.split('\t', 1)  # split id from text on first tab
                rows.append({
                    'ID': id_.strip(),
                    'Text': text.strip(),
                    'valence': int(valence.strip()),
                    'arousal': int(arousal.strip()),
                    'deva_label': label.strip(),
                    'dataset': dataset_label
                })
    return pd.DataFrame(rows)


In [108]:
df_deva_gh = parse_deva_output(DEVA_OUTPUT_DIR / "gs_gh_out.txt", "gs_gh")
df_deva_gh.head(5)

,ID,Text,valence,arousal,deva_label,dataset
0,329914,Hmm... Very mysterious development. Not be eas...,1,2,NEUTRAL (HA),gs_gh
1,2728032,:+1:,1,1,NEUTRAL,gs_gh
2,3098507,So what I'm hearing is that if I: * tag consis...,1,-3,NEUTRAL (LA),gs_gh
3,3103230,How did you tested by normal player? Have you ...,-3,-4,DEPRESSED,gs_gh
4,95688,Nice indeed. An earlier version of this patch ...,4,-3,RELAXED,gs_gh


In [109]:
df_deva_so = parse_deva_output(DEVA_OUTPUT_DIR / "gs_so_out.txt", "gs_so")
df_deva_so.head(5)

,ID,Text,valence,arousal,deva_label,dataset
0,25607311,I saw a really nice post on stackoverflow on t...,4,-3,RELAXED,gs_so
1,25386947,I tried so many different ways for printing wi...,4,-4,RELAXED,gs_so
2,25419608,I have a general style definition as Now want ...,3,-3,RELAXED,gs_so
3,25727724,My category.php uses wp_list_categories to dis...,1,-3,NEUTRAL (LA),gs_so
4,25346200,I'm trying to test my controller which deal wi...,-2,-3,DEPRESSED,gs_so


In [110]:
df_deva_cou = parse_deva_output(DEVA_OUTPUT_DIR / "cou_out.txt", "cou")
df_deva_cou.head(5)

,ID,Text,valence,arousal,deva_label,dataset
0,0,did you get a chance to follow up on the issue...,1,-2,NEUTRAL (LA),cou
1,1,"""Thank you for this PR. This is an interestin...",-3,-4,DEPRESSED,cou
2,2,", oh sorry for this, my fault, i'll be more c...",-2,-3,DEPRESSED,cou
3,3,I've added so that we can add documentation r...,1,-3,NEUTRAL (LA),cou
4,4,Applications can still use or qualifiers to g...,-2,-4,DEPRESSED,cou


## Mapping DEVA results to polarity labels

As mentioned above, the following mapping is used:
- valence > +1 -> positive
- valence < -1 -> negative
- valence = ± 1 -> neutral

In [79]:
def valence_to_polarity(v):
    if v > 1:
        return 'positive'
    elif v < -1:
        return 'negative'
    else:
        return 'neutral'

In [83]:
df_deva_gh['deva_polarity'] = df_deva_gh['valence'].apply(valence_to_polarity)
df_deva_gh

,ID,Text,valence,arousal,deva_label,dataset,deva_polarity
0,329914,Hmm... Very mysterious development. Not be eas...,1,2,NEUTRAL (HA),gs_gh,neutral
1,2728032,:+1:,1,1,NEUTRAL,gs_gh,neutral
2,3098507,So what I'm hearing is that if I: * tag consis...,1,-3,NEUTRAL (LA),gs_gh,neutral
3,3103230,How did you tested by normal player? Have you ...,-3,-4,DEPRESSED,gs_gh,negative
4,95688,Nice indeed. An earlier version of this patch ...,4,-3,RELAXED,gs_gh,positive
...,...,...,...,...,...,...,...
393,739112,Can clarify this with `Hash#fetch`: ``` second...,1,-4,NEUTRAL (LA),gs_gh,neutral
394,1217254,It does: Endianess is Netty (and in Java gener...,1,-3,NEUTRAL (LA),gs_gh,neutral
395,1581644,In the spirit of testing all the guarantees in...,3,5,EXCITED,gs_gh,positive
396,3991765,More proper would be: ```php if (self::$truste...,1,-3,NEUTRAL (LA),gs_gh,neutral


In [84]:
df_deva_so['deva_polarity'] = df_deva_so['valence'].apply(valence_to_polarity)
df_deva_so

,ID,Text,valence,arousal,deva_label,dataset,deva_polarity
0,25607311,I saw a really nice post on stackoverflow on t...,4,-3,RELAXED,gs_so,positive
1,25386947,I tried so many different ways for printing wi...,4,-4,RELAXED,gs_so,positive
2,25419608,I have a general style definition as Now want ...,3,-3,RELAXED,gs_so,positive
3,25727724,My category.php uses wp_list_categories to dis...,1,-3,NEUTRAL (LA),gs_so,neutral
4,25346200,I'm trying to test my controller which deal wi...,-2,-3,DEPRESSED,gs_so,negative
...,...,...,...,...,...,...,...
196,25420741,I have written a code in java for web services...,1,-4,NEUTRAL (LA),gs_so,neutral
197,25695655,Under the code that i'm trying to use for add ...,-2,-3,DEPRESSED,gs_so,negative
198,25614128,"According to the manual, it says . In my opini...",1,-3,NEUTRAL (LA),gs_so,neutral
199,25477108,I am designing a system that will need to deal...,4,-4,RELAXED,gs_so,positive


In [85]:
df_deva_cou['deva_polarity'] = df_deva_cou['valence'].apply(valence_to_polarity)
df_deva_cou

,ID,Text,valence,arousal,deva_label,dataset,deva_polarity
0,0,did you get a chance to follow up on the issue...,1,-2,NEUTRAL (LA),cou,neutral
1,1,"""Thank you for this PR. This is an interestin...",-3,-4,DEPRESSED,cou,negative
2,2,", oh sorry for this, my fault, i'll be more c...",-2,-3,DEPRESSED,cou,negative
3,3,I've added so that we can add documentation r...,1,-3,NEUTRAL (LA),cou,neutral
4,4,Applications can still use or qualifiers to g...,-2,-4,DEPRESSED,cou,negative
...,...,...,...,...,...,...,...
1786,1786,"Hmm, looks like r737027 corresponds to version...",1,-3,NEUTRAL (LA),cou,neutral
1787,1787,"<!-- First of all, thank you for your contribu...",-2,5,STRESSED,cou,negative
1788,1788,"there was a typo in the comment, also made a s...",1,-3,NEUTRAL (LA),cou,neutral
1789,1789,Fixed some typos. Added the ghost button type....,-2,-4,DEPRESSED,cou,negative


## Comparing resulting DEVA labels to original labels

In [86]:
merged_gh = df_deva_gh.assign(ID=df_deva_gh['ID'].astype(int)).merge(
    df_gs_gh[['ID', 'Gold label']],
    on='ID'
).rename(columns={'Gold label': 'gold_label'})

merged_gh

,ID,Text,valence,arousal,deva_label,dataset,deva_polarity,gold_label
0,329914,Hmm... Very mysterious development. Not be eas...,1,2,NEUTRAL (HA),gs_gh,neutral,negative
1,2728032,:+1:,1,1,NEUTRAL,gs_gh,neutral,neutral
2,3098507,So what I'm hearing is that if I: * tag consis...,1,-3,NEUTRAL (LA),gs_gh,neutral,neutral
3,3103230,How did you tested by normal player? Have you ...,-3,-4,DEPRESSED,gs_gh,negative,positive
4,95688,Nice indeed. An earlier version of this patch ...,4,-3,RELAXED,gs_gh,positive,positive
...,...,...,...,...,...,...,...,...
393,739112,Can clarify this with `Hash#fetch`: ``` second...,1,-4,NEUTRAL (LA),gs_gh,neutral,neutral
394,1217254,It does: Endianess is Netty (and in Java gener...,1,-3,NEUTRAL (LA),gs_gh,neutral,neutral
395,1581644,In the spirit of testing all the guarantees in...,3,5,EXCITED,gs_gh,positive,neutral
396,3991765,More proper would be: ```php if (self::$truste...,1,-3,NEUTRAL (LA),gs_gh,neutral,neutral


In [87]:
merged_so = df_deva_so.assign(ID=df_deva_so['ID'].astype(int)).merge(
    df_gs_so[['ID', 'Gold label']],
    on='ID'
).rename(columns={'Gold label': 'gold_label'})

merged_so

,ID,Text,valence,arousal,deva_label,dataset,deva_polarity,gold_label
0,25607311,I saw a really nice post on stackoverflow on t...,4,-3,RELAXED,gs_so,positive,positive
1,25386947,I tried so many different ways for printing wi...,4,-4,RELAXED,gs_so,positive,negative
2,25419608,I have a general style definition as Now want ...,3,-3,RELAXED,gs_so,positive,neutral
3,25727724,My category.php uses wp_list_categories to dis...,1,-3,NEUTRAL (LA),gs_so,neutral,neutral
4,25346200,I'm trying to test my controller which deal wi...,-2,-3,DEPRESSED,gs_so,negative,neutral
...,...,...,...,...,...,...,...,...
196,25420741,I have written a code in java for web services...,1,-4,NEUTRAL (LA),gs_so,neutral,negative
197,25695655,Under the code that i'm trying to use for add ...,-2,-3,DEPRESSED,gs_so,negative,mixed
198,25614128,"According to the manual, it says . In my opini...",1,-3,NEUTRAL (LA),gs_so,neutral,neutral
199,25477108,I am designing a system that will need to deal...,4,-4,RELAXED,gs_so,positive,neutral


In [104]:
df_cou_labels = df_cou[['gold_label']].copy()
df_cou_labels.insert(0, 'ID', df_cou_labels.index.astype(str))
df_cou_labels = df_cou_labels[df_cou_labels['gold_label'].notna() & (df_cou_labels['gold_label'] != 'undefined')]

merged_cou = df_deva_cou.merge(df_cou_labels, on='ID')
merged_cou

,ID,Text,valence,arousal,deva_label,dataset,deva_polarity,gold_label
0,0,did you get a chance to follow up on the issue...,1,-2,NEUTRAL (LA),cou,neutral,neutral
1,1,"""Thank you for this PR. This is an interestin...",-3,-4,DEPRESSED,cou,negative,neutral
2,2,", oh sorry for this, my fault, i'll be more c...",-2,-3,DEPRESSED,cou,negative,negative
3,3,I've added so that we can add documentation r...,1,-3,NEUTRAL (LA),cou,neutral,neutral
4,4,Applications can still use or qualifiers to g...,-2,-4,DEPRESSED,cou,negative,negative
...,...,...,...,...,...,...,...,...
1786,1786,"Hmm, looks like r737027 corresponds to version...",1,-3,NEUTRAL (LA),cou,neutral,negative
1787,1787,"<!-- First of all, thank you for your contribu...",-2,5,STRESSED,cou,negative,neutral
1788,1788,"there was a typo in the comment, also made a s...",1,-3,NEUTRAL (LA),cou,neutral,neutral
1789,1789,Fixed some typos. Added the ghost button type....,-2,-4,DEPRESSED,cou,negative,neutral


## Calculating precision, recall, F1 for DEVA results

For each class $cls$ of $classes=[positive, negative, neutral]$, we compute:
$$\text{precision}(cls) = \frac{TP}{TP + FP}$$
$$\text{recall}(cls) = \frac{TP}{TP + FN}$$
$$\text{f1}(cls) = 2\times\frac{\text{precision}(cls) \times \text{recall}(cls)}{\text{precision}(cls) + \text{recall}(cls)},$$
where for each class $cls$, TP is the number of messages correctly predicted as $cls$, FP is the number of messages incorrectly predicted as $cls$, and FN is the number of messages that belong to $cls$ but were predicted as a different class.

Moreover, for each metric, we compute macro averages and weighted averages:
$$\text{precision}_{\text{macro}} = \frac{1}{|classses|}\sum_{cls}\text{precision}(cls)$$
$$\text{precision}_{\text{weighted}} = \frac{\sum_{cls}\text{precision}(cls)\times|cls|}{\sum_{cls}|cls|}$$

In [119]:
def compute_metrics(df, dataset_name):
    classes = ['positive', 'negative', 'neutral']
    rows = []

    for cls in classes:
        tp = ((df['deva_polarity'] == cls) & (df['gold_label'] == cls)).sum()
        fp = ((df['deva_polarity'] == cls) & (df['gold_label'] != cls)).sum()
        fn = ((df['deva_polarity'] != cls) & (df['gold_label'] == cls)).sum()

        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        recall    = tp / (tp + fn) if (tp + fn) > 0 else 0
        f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

        rows.append({
            'dataset': dataset_name,
            'class': cls,
            'precision': precision,
            'recall': recall,
            'f1': f1,
            'support': (df['gold_label'] == cls).sum()
        })

    # macro average (simple mean across classes, like Coutinho et al.)
    macro = pd.DataFrame(rows)
    rows.append({
        'dataset': dataset_name,
        'class': 'macro avg',
        'precision': macro['precision'].mean(),
        'recall': macro['recall'].mean(),
        'f1': macro['f1'].mean(),
        'support': len(df)
    })

    # weighted average (mean weighted by class support)
    total = len(df)
    rows.append({
        'dataset': dataset_name,
        'class': 'weighted avg',
        'precision': (macro['precision'] * macro['support']).sum() / total,
        'recall':    (macro['recall']    * macro['support']).sum() / total,
        'f1':        (macro['f1']        * macro['support']).sum() / total,
        'support': total
    })

    return pd.DataFrame(rows)

In [120]:
results_gh  = compute_metrics(merged_gh,  'gs_gh')
results_so  = compute_metrics(merged_so,  'gs_so')
results_cou = compute_metrics(merged_cou, 'cou')

results_all = pd.concat([results_gh, results_so, results_cou], ignore_index=True)
results_all

,dataset,class,precision,recall,f1,support
0,gs_gh,positive,0.492958,0.673077,0.569106,52
1,gs_gh,negative,0.194444,0.297872,0.235294,47
2,gs_gh,neutral,0.886275,0.779310,0.829358,290
3,gs_gh,macro avg,0.524559,0.583420,0.544586,398
4,gs_gh,weighted avg,0.733146,0.690955,0.706447,398
5,gs_so,positive,0.135135,0.500000,0.212766,20
6,gs_so,negative,0.261905,0.314286,0.285714,35
7,gs_so,neutral,0.847059,0.545455,0.663594,132
8,gs_so,macro avg,0.414700,0.453247,0.387358,201
9,gs_so,weighted avg,0.615329,0.462687,0.506715,201


In [121]:
# Format results
results_display = results_all.copy()
for col in ['precision', 'recall', 'f1']:
    results_display[col] = results_display[col].map(lambda x: f"{x:.2%}")

print(results_display.to_string(index=False))

dataset        class precision recall     f1  support
  gs_gh     positive    49.30% 67.31% 56.91%       52
  gs_gh     negative    19.44% 29.79% 23.53%       47
  gs_gh      neutral    88.63% 77.93% 82.94%      290
  gs_gh    macro avg    52.46% 58.34% 54.46%      398
  gs_gh weighted avg    73.31% 69.10% 70.64%      398
  gs_so     positive    13.51% 50.00% 21.28%       20
  gs_so     negative    26.19% 31.43% 28.57%       35
  gs_so      neutral    84.71% 54.55% 66.36%      132
  gs_so    macro avg    41.47% 45.32% 38.74%      201
  gs_so weighted avg    61.53% 46.27% 50.67%      201
    cou     positive    66.49% 72.36% 69.30%      521
    cou     negative    48.41% 70.37% 57.36%      432
    cou      neutral    79.87% 56.80% 66.39%      838
    cou    macro avg    64.92% 66.51% 64.35%     1791
    cou weighted avg    68.39% 64.60% 65.06%     1791


Note: the weighted average in the results is higher than the macro average mostly because the model seems to do much better on neutral text